# Canadian Market Risk Snapshot
Volatility, correlation, Value at Risk and stress tests for Canadian stocks (XIU), bonds (ZAG) and gold (CGL), using one year of daily closing prices (6 Oct 2025 to 2 Oct 2026).

Two portfolios are compared: **Equal Weight** (one third each) and **Conservative** (30% stocks, 60% bonds, 10% gold). Keep `SCENARIO_ANALYSIS_formatted.xlsx` in the same folder as this notebook.

## Step 1: Load the prices
Read the date and closing price of each asset from the Excel file.

In [1]:
import numpy as np
import pandas as pd

FILE = "SCENARIO_ANALYSIS_formatted.xlsx"

raw = pd.read_excel(FILE, header=None, skiprows=5, usecols="B,C,G,H,L,M")
prices = pd.DataFrame({
    "Date": raw.iloc[:, 0],
    "Stocks": raw.iloc[:, 1],
    "Bonds": raw.iloc[:, 3],
    "Gold": raw.iloc[:, 5],
}).set_index("Date")

print(prices.head())

                     Stocks  Bonds   Gold
Date                                     
2025-10-06 16:00:00   45.52  13.91  30.73
2025-10-07 16:00:00   45.27  13.92  30.83
2025-10-08 16:00:00   45.34  13.92  31.40
2025-10-09 16:00:00   45.08  13.91  30.83
2025-10-10 16:00:00   44.45  13.95  31.15


## Step 2: Daily returns
Daily return = today's price / yesterday's price - 1.

In [2]:
returns = prices.pct_change().dropna()
print(returns.head())

                       Stocks     Bonds      Gold
Date                                             
2025-10-07 16:00:00 -0.005492  0.000719  0.003254
2025-10-08 16:00:00  0.001546  0.000000  0.018488
2025-10-09 16:00:00 -0.005734 -0.000718 -0.018153
2025-10-10 16:00:00 -0.013975  0.002876  0.010380
2025-10-14 16:00:00  0.015298  0.000717  0.032745


## Step 3: Volatility
Volatility is the standard deviation of daily returns. Multiplying by the square root of 252 trading days gives the annual figure.

In [3]:
daily_vol = returns.std()
annual_vol = daily_vol * np.sqrt(252)

print((daily_vol * 100).round(2))
print((annual_vol * 100).round(2))

Stocks    0.79
Bonds     0.30
Gold      1.89
dtype: float64
Stocks    12.46
Bonds      4.74
Gold      30.08
dtype: float64


## Step 4: Correlation
Correlation shows how much two assets move together (1 = always together, 0 = unrelated, negative = opposite).

In [4]:
corr = returns.corr()
print(corr.round(2))

        Stocks  Bonds  Gold
Stocks    1.00   0.41  0.56
Bonds     0.41   1.00  0.19
Gold      0.56   0.19  1.00


## Step 5: Build two portfolios
The portfolio return each day is the weighted sum of the three asset returns.

In [5]:
returns["Equal Weight"] = returns[["Stocks", "Bonds", "Gold"]].mean(axis=1)
returns["Conservative"] = 0.3*returns["Stocks"] + 0.6*returns["Bonds"] + 0.1*returns["Gold"]

port_vol = returns[["Equal Weight", "Conservative"]].std() * np.sqrt(252)
print((port_vol * 100).round(2))

Equal Weight    13.37
Conservative     7.46
dtype: float64


## Step 6: Value at Risk (VaR)
95% daily VaR on a $100,000 portfolio, calculated two ways: parametric (1.645 x daily volatility) and historical (the 5th percentile of actual daily returns). The worst single day is shown for comparison.

In [6]:
value = 100000   # pretend portfolio size in CAD
cols = ["Stocks", "Equal Weight", "Conservative"]

var_param = 1.645 * returns[cols].std() * value
var_hist = -returns[cols].quantile(0.05) * value
worst_day = -returns[cols].min() * value

print("Parametric VaR:\n", var_param.round(0))
print("Historical VaR:\n", var_hist.round(0))
print("Worst day:\n", worst_day.round(0))

Parametric VaR:
 Stocks          1292.0
Equal Weight    1385.0
Conservative     774.0
dtype: float64
Historical VaR:
 Stocks          1342.0
Equal Weight    1382.0
Conservative     756.0
Name: 0.05, dtype: float64
Worst day:
 Stocks          2818.0
Equal Weight    4275.0
Conservative    1810.0
dtype: float64


## Step 7: Stress test
Four what-if scenarios applied to each portfolio. The shocks are assumptions, not forecasts. For example, the 7.5% bond loss in the rates scenario assumes a bond fund duration of about 7.5 years.

In [7]:
scenarios = pd.DataFrame({
    "Stocks": [-0.03, -0.15, -0.03, -0.12],
    "Bonds":  [-0.075, 0.02, 0.0, -0.05],
    "Gold":   [-0.02, -0.08, -0.15, -0.10]},
    index=["Rates +1%", "Stock correction", "Gold falls", "Everything falls"])

weights = pd.DataFrame({
    "Stocks only":  [1, 0, 0],
    "Equal Weight": [1/3, 1/3, 1/3],
    "Conservative": [0.3, 0.6, 0.1]},
    index=["Stocks", "Bonds", "Gold"])

pnl = scenarios.dot(weights) * value
print(pnl.round(0))

                  Stocks only  Equal Weight  Conservative
Rates +1%             -3000.0       -4167.0       -5600.0
Stock correction     -15000.0       -7000.0       -4100.0
Gold falls            -3000.0       -6000.0       -2400.0
Everything falls     -12000.0       -9000.0       -7600.0


Conservative loses the most when rates rise, because 60% of it is in bonds. It loses the least in the other three scenarios.

## Step 8: Save the returns to a SQL database
The returns are stored in a SQLite database file (`risk.db`) so they can be queried with SQL.

In [8]:
import sqlite3

conn = sqlite3.connect("risk.db")

r = returns.copy()
r.index = r.index.strftime("%Y-%m-%d")
r.index.name = "date"
r.columns = r.columns.str.replace(" ", "_")
r.to_sql("daily_returns", conn, if_exists="replace")

def q(sql):
    return pd.read_sql(sql, conn)

## Step 9: Risk questions in SQL
First, check the table loaded correctly.

In [9]:
q("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,daily_returns


In [10]:
q("SELECT * FROM daily_returns LIMIT 5")

,date,Stocks,Bonds,Gold,Equal_Weight,Conservative
0,2025-10-07,-0.005492,0.000719,0.003254,-0.000506,-0.000891
1,2025-10-08,0.001546,0.000000,0.018488,0.006678,0.002313
2,2025-10-09,-0.005734,-0.000718,-0.018153,-0.008202,-0.003967
3,2025-10-10,-0.013975,0.002876,0.010380,-0.000240,-0.001429
4,2025-10-14,0.015298,0.000717,0.032745,0.016253,0.008294


In [11]:
q("SELECT COUNT(*) AS total_days FROM daily_returns")

,total_days
0,249


**What were the five worst days for stocks?**

In [12]:
q("SELECT date, ROUND(Stocks*100,2) AS stocks_pct FROM daily_returns ORDER BY Stocks LIMIT 5")

,date,stocks_pct
0,2026-01-30,-2.82
1,2026-06-05,-1.93
2,2026-03-03,-1.80
3,2026-02-12,-1.77
4,2025-11-13,-1.76


**Which days did stocks fall by more than 1%?**

In [13]:
q("SELECT date, Stocks FROM daily_returns WHERE Stocks < -0.01")

,date,Stocks
0,2025-10-10,-0.013975
1,2025-10-21,-0.013699
2,2025-10-29,-0.010999
3,2025-11-04,-0.015044
4,2025-11-13,-0.017613
5,2026-01-20,-0.014183
6,2026-01-30,-0.028181
7,2026-02-05,-0.012167
8,2026-02-12,-0.017677
9,2026-03-03,-0.017978


The same question, passing the value safely with a `?` parameter instead of typing it into the SQL text:

In [14]:
pd.read_sql("SELECT date, Stocks FROM daily_returns WHERE Stocks < ?",
            conn, params=(-0.01,))

,date,Stocks
0,2025-10-10,-0.013975
1,2025-10-21,-0.013699
2,2025-10-29,-0.010999
3,2025-11-04,-0.015044
4,2025-11-13,-0.017613
5,2026-01-20,-0.014183
6,2026-01-30,-0.028181
7,2026-02-05,-0.012167
8,2026-02-12,-0.017677
9,2026-03-03,-0.017978


**What was the average daily stock return in each month?** (This is the average daily return, not the monthly total.)

In [15]:
q("""
SELECT substr(date,1,7) AS month,
       ROUND(AVG(Stocks)*100,2) AS avg_stocks_pct
FROM daily_returns
GROUP BY month
ORDER BY month
""")

,month,avg_stocks_pct
0,2025-10,-0.04
1,2025-11,0.14
2,2025-12,0.06
3,2026-01,-0.01
4,2026-02,0.31
5,2026-03,-0.14
6,2026-04,0.20
7,2026-05,0.07
8,2026-06,0.09
9,2026-07,0.08


**On how many days did stocks, bonds and gold all fall?**

In [16]:
q("SELECT COUNT(*) AS days_all_fell FROM daily_returns WHERE Stocks<0 AND Bonds<0 AND Gold<0")

,days_all_fell
0,34


**When stocks fell, did bonds protect the portfolio?**

In [17]:
q("""
SELECT COUNT(*) AS total_days,
       SUM(Stocks<0) AS stock_down,
       SUM(Stocks<0 AND Bonds<0) AS stock_bond_down,
       SUM(Stocks<0 AND Bonds>0) AS stock_down_bond_up
FROM daily_returns
""")

,total_days,stock_down,stock_bond_down,stock_down_bond_up
0,249,102,57,36


## Key takeaways
- Annual volatility: bonds 4.74%, stocks 12.46%, gold 30.08%.
- Correlations are all positive (0.19 to 0.56), so diversification helps only partly. On the 102 days stocks fell, bonds rose on only 36.
- Equal Weight (13.37% volatility) is riskier than stocks alone because of gold. Conservative is the lowest at 7.46%.
- In the stress test, Conservative loses the most when rates rise, so low volatility does not mean safe in every scenario.

Limitations: one year of price data (no dividends or interest), VaR assumes the recent past represents the future, and the stress-test shocks are chosen assumptions.